# 📈 Visualisation et Exploration des Données (EDA)
Ce notebook explore les datasets pour isoler les facteurs influençant les **rendements agricoles**.
L'objectif final est de prédire le rendement (target), nous allons donc vérifier quelques hypothèses :
1. Le rendement a-t-il augmenté au fil du temps (technologie, etc.) ?
2. Quel est l'impact du climat (températures et précipitations) ?
3. L'utilisation de pesticides et d'engrais montre-t-elle une forte relation avec le rendement ?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
sns.set_palette('Set2')
DIR = 'data/cleaned'


## 1. Chargement et préparation des données

In [ ]:
# Chargement de la cible : Rendement (Cultures végétales)
df_cultures = pd.read_csv(f'{DIR}/production_cultures.csv')
df_rendement = df_cultures[df_cultures['Element'] == 'Rendement'].copy()
df_rendement = df_rendement.rename(columns={'Valeur': 'Rendement_kg_ha'})

# Température
df_temp = pd.read_csv(f'{DIR}/mean_temperature.csv')
df_temp = df_temp.rename(columns={'Valeur': 'Temperature_C'})

# Précipitations
df_precip = pd.read_csv(f'{DIR}/precipitations.csv')
df_precip = df_precip.rename(columns={'Valeur': 'Precipitations_mm'})

# Pesticides (Agrégation par pays/année)
df_pest = pd.read_csv(f'{DIR}/pesticides.csv')
df_pest_agg = df_pest.groupby(['Pays', 'Annee'])['Valeur'].sum().reset_index()
df_pest_agg = df_pest_agg.rename(columns={'Valeur': 'Pesticides_kg_ha'})

# Engrais (Fertilizers)
df_fert = pd.read_csv(f'{DIR}/fertilizers_nutrient.csv')
df_fert_agg = df_fert.groupby(['Pays', 'Annee'])['Valeur'].sum().reset_index()
df_fert_agg = df_fert_agg.rename(columns={'Valeur': 'Engrais_kg_ha'})
print("Datasets source chargés avec succès.")

KeyError: 'Element'

## 2. Aperçu des Distributions et Classements

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_rendement['Rendement_kg_ha'], bins=50, kde=True, color='purple')
plt.title('Distribution Globale des Rendements Agricoles', fontsize=14)
plt.xlabel('Rendement (kg/ha)')
plt.ylabel('Fréquence')
plt.xscale('log')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_crops = df_rendement.groupby('Produit')['Rendement_kg_ha'].mean().nlargest(10)
sns.barplot(x=top_crops.values, y=top_crops.index, ax=axes[0], palette='viridis')
axes[0].set_title('Top 10 des Cultures par Rendement Moyen')
axes[0].set_xlabel('Rendement Moyen (kg/ha)')

top_countries = df_rendement.groupby('Pays')['Rendement_kg_ha'].mean().nlargest(10)
sns.barplot(x=top_countries.values, y=top_countries.index, ax=axes[1], palette='magma')
axes[1].set_title('Top 10 des Pays par Rendement Moyen')
axes[1].set_xlabel('Rendement Moyen (kg/ha)')

plt.tight_layout()
plt.show()


### 🔄 Fusion des données pour les Graphiques de relation

In [ ]:
# On calcule le rendement moyen global par pays/année
df_rend_agg = df_rendement.groupby(['Pays', 'Annee'])['Rendement_kg_ha'].mean().reset_index()

# Jointures
df_merged = df_rend_agg.merge(df_temp[['Pays', 'Annee', 'Temperature_C']], on=['Pays', 'Annee'], how='inner')
df_merged = df_merged.merge(df_precip[['Pays', 'Annee', 'Precipitations_mm']], on=['Pays', 'Annee'], how='inner')
df_merged = df_merged.merge(df_pest_agg, on=['Pays', 'Annee'], how='left')
df_merged = df_merged.merge(df_fert_agg, on=['Pays', 'Annee'], how='left')

display(df_merged.head())
print(f"Taille du dataset fusionné : {df_merged.shape}")


## 3. Évolution Globale des Rendements au fil du temps


In [ ]:
trend_yearly = df_rendement.groupby('Annee')['Rendement_kg_ha'].mean()

plt.figure(figsize=(14, 5))
plt.plot(trend_yearly.index, trend_yearly.values, marker='o', color='#2ca02c', linewidth=2)
plt.title('Évolution du Rendement Agricole Moyen Mondial', fontsize=14, pad=15)
plt.xlabel('Année')
plt.ylabel('Rendement moyen (kg/ha)')
plt.show()


## 4. Impact du Climat sur le Rendement (Température & Précipitations)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.regplot(data=df_merged, x='Temperature_C', y='Rendement_kg_ha', 
            scatter_kws={'alpha':0.3, 'color':'teal'}, line_kws={'color':'red', 'linewidth': 2}, ax=axes[0])
axes[0].set_title('Rendement vs Température Moyenne (°C)', pad=10)
axes[0].set_xlabel('Température (°C)')
axes[0].set_ylabel('Rendement Moyen (kg/ha)')

sns.regplot(data=df_merged, x='Precipitations_mm', y='Rendement_kg_ha', 
            scatter_kws={'alpha':0.3, 'color':'steelblue'}, line_kws={'color':'red', 'linewidth': 2}, ax=axes[1])
axes[1].set_title('Rendement vs Précipitations (mm)', pad=10)
axes[1].set_xlabel('Précipitations (mm)')
axes[1].set_ylabel('Rendement Moyen (kg/ha)')

plt.tight_layout()
plt.show()


## 5. Impact des Intrants : Pesticides & Engrais


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.regplot(data=df_merged[df_merged['Pesticides_kg_ha'] > 0], 
            x='Pesticides_kg_ha', y='Rendement_kg_ha', 
            scatter_kws={'alpha':0.4, 'color':'#d62728'}, line_kws={'color':'black', 'linewidth': 2}, ax=axes[0])
axes[0].set_title('Rendement en fonction des Pesticides')
axes[0].set_xscale('log')
axes[0].set_xlabel('Pesticides (log kg/ha)')
axes[0].set_ylabel('Rendement')

sns.regplot(data=df_merged[df_merged['Engrais_kg_ha'] > 0], 
            x='Engrais_kg_ha', y='Rendement_kg_ha', 
            scatter_kws={'alpha':0.4, 'color':'#ff7f0e'}, line_kws={'color':'black', 'linewidth': 2}, ax=axes[1])
axes[1].set_title('Rendement en fonction des Engrais')
axes[1].set_xscale('log')
axes[1].set_xlabel('Engrais (log kg/ha)')
axes[1].set_ylabel('Rendement')

plt.tight_layout()
plt.show()


## 6. Analyse Multivariée Globale (Pairplot & Matrice)


In [ ]:
# Un Pairplot permet de voir toutes les intéractions d'un coup (on prend un sample pour la rapidité)
sample_df = df_merged.dropna().sample(n=min(1000, len(df_merged.dropna())), random_state=42)
sns.pairplot(sample_df[['Rendement_kg_ha', 'Temperature_C', 'Precipitations_mm', 'Pesticides_kg_ha']], 
             diag_kind='kde', corner=True, plot_kws={'alpha': 0.5, 'color': 'goldenrod'})
plt.suptitle('Interactions Croisées entre Rendement et Variables Environnementales', y=1.02, fontsize=14)
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
corr_cols = ['Rendement_kg_ha', 'Temperature_C', 'Precipitations_mm', 'Pesticides_kg_ha', 'Engrais_kg_ha']
corr_matrix = df_merged[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, linewidths=.5, fmt='.2f')
plt.title('Matrice de Corrélation de Pearson', fontsize=16, pad=15)
plt.show()


### ✍️ Conclusions pour le Machine Learning

1. **Distributions variées** : Le rendement est très asymétrique selon les pays et les récoltes.
2. **Le paradoxe du climat** : Le climat a beaucoup de variance. Un modèle purement linéaire sera médiocre. Il faudra privilégier des modèles à arbres décisionnels (Random Forest, XGBoost) car un surplus de température ou de pluie est destructeur ! 
3. **L'importance pivot des intrants** : Engrais et Pesticides ont généralement la corrélation positive la plus constante avec le taux de Rendement.
4. **Série temporelle** : La composante « Année » capte la tendance à la hausse technologique globale.

**L'étape suivante consiste à concevoir le pipeline d'entraînement !**